# Module 07: Random Projections & the Johnson-Lindenstrauss Lemma

**Time**: ~30 minutes

**Prerequisites**: Modules 00-06 (PyTorch basics, WHT, random rotations, Lloyd-Max, MSE quantizer)

---

We have just finished building the full MSE-optimal quantizer (Stage 1 of TurboQuant). Now we start a **completely independent thread** -- Stage 2 -- which attacks the problem from a different angle.

The question: can we compress vectors by projecting them into **fewer dimensions** while preserving the geometric relationships we care about?

In [ ]:
import sys; sys.path.insert(0, "..")
from src.helpers import check_intuition, section_header

import math
import torch
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

## 1. The Dimension Problem

Embeddings are high-dimensional: 128, 768, 1536 dimensions are common. When we need to compute **pairwise distances** (for nearest-neighbor search, attention, clustering), the cost is:

- Storing all $n$ vectors: $O(n \cdot d)$ memory
- Computing one distance: $O(d)$ operations
- All pairwise distances: $O(n^2 \cdot d)$ operations

The question is: **can we project to $m \ll d$ dimensions and still preserve distances?**

If so, we'd save a factor of $d/m$ on every distance computation.

In [ ]:
section_header(
    "The Dimension Problem",
    "How expensive is high-dimensional geometry?"
)

# Typical embedding dimensions and their costs
dims = [128, 768, 1536, 4096]
n_vectors = 100_000

print(f"For n = {n_vectors:,} vectors:")
print(f"{'Dimension':>10}  {'Storage (MB)':>14}  {'Pairwise FLOPs':>16}")
print("-" * 46)
for d in dims:
    storage_mb = n_vectors * d * 4 / 1e6  # float32
    pairwise_flops = n_vectors**2 * d
    print(f"{d:>10}  {storage_mb:>14,.1f}  {pairwise_flops:>16.2e}")

## 2. Random Projection -- The Surprising Idea

Here is the plan:

1. Generate a random matrix $R$ of shape $(m, d)$ where $m \ll d$
2. Project every vector: $y = \frac{1}{\sqrt{m}} R \, x$
3. **Claim**: distances between projected points $\approx$ distances between original points

This seems too good to be true. A random matrix, with no knowledge of our data, somehow preserves the geometry? Let's test it.

In [ ]:
section_header(
    "Random Projection -- The Surprising Idea",
    "Project with a random matrix, preserve distances"
)

torch.manual_seed(42)

d = 512          # original dimension
m = 50           # projected dimension (10x compression!)
n = 200          # number of vectors

# Create random data vectors
X = torch.randn(n, d)

# Create a random Gaussian projection matrix
R = torch.randn(m, d) / math.sqrt(m)   # the 1/sqrt(m) scaling is key

# Project
Y = X @ R.t()    # shape: (n, m)

print(f"Original shape: {X.shape}  -->  Projected shape: {Y.shape}")
print(f"Compression ratio: {d / m:.0f}x fewer dimensions")

Now let's compare **all pairwise distances** before and after projection.

In [ ]:
# Compute pairwise distances in original and projected spaces
true_dists = torch.cdist(X, X)          # (n, n)
proj_dists = torch.cdist(Y, Y)          # (n, n)

# Extract upper triangle (unique pairs)
mask = torch.triu(torch.ones(n, n, dtype=torch.bool), diagonal=1)
true_flat = true_dists[mask].numpy()
proj_flat = proj_dists[mask].numpy()

# Subsample for plotting (19900 points is a lot)
rng = np.random.default_rng(0)
idx = rng.choice(len(true_flat), size=min(3000, len(true_flat)), replace=False)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=true_flat[idx], y=proj_flat[idx],
    mode="markers",
    marker=dict(size=3, opacity=0.4, color="steelblue"),
    name="Pairs",
))

# Perfect preservation line
lo, hi = true_flat.min(), true_flat.max()
fig.add_trace(go.Scatter(
    x=[lo, hi], y=[lo, hi],
    mode="lines", line=dict(color="red", dash="dash", width=2),
    name="Perfect",
))

corr = np.corrcoef(true_flat, proj_flat)[0, 1]
fig.update_layout(
    title=f"Pairwise Distances: d={d} -> m={m}  (correlation = {corr:.4f})",
    xaxis_title="True distance (d=512)",
    yaxis_title="Projected distance (m=50)",
    template="plotly_white",
    height=500, width=600,
)
fig.show()

That correlation is **very** tight! A random matrix with no training, no data-dependence, and 10x fewer dimensions preserves the distance structure remarkably well.

### How does quality change with the projection dimension $m$?

Let's try $m \in \{10, 25, 50, 100, 200\}$ and see.

In [ ]:
m_values = [10, 25, 50, 100, 200]

fig = make_subplots(
    rows=1, cols=len(m_values),
    subplot_titles=[f"m = {m_val}" for m_val in m_values],
    horizontal_spacing=0.04,
)

for i, m_val in enumerate(m_values, 1):
    R_i = torch.randn(m_val, d) / math.sqrt(m_val)
    Y_i = X @ R_i.t()
    proj_d = torch.cdist(Y_i, Y_i)[mask].numpy()

    sample = rng.choice(len(true_flat), size=min(2000, len(true_flat)), replace=False)
    corr_i = np.corrcoef(true_flat, proj_d)[0, 1]

    fig.add_trace(
        go.Scatter(
            x=true_flat[sample], y=proj_d[sample],
            mode="markers",
            marker=dict(size=2, opacity=0.3, color="steelblue"),
            showlegend=False,
        ),
        row=1, col=i,
    )
    fig.add_trace(
        go.Scatter(
            x=[lo, hi], y=[lo, hi],
            mode="lines", line=dict(color="red", dash="dash", width=1),
            showlegend=False,
        ),
        row=1, col=i,
    )

    # Annotate with correlation
    fig.add_annotation(
        text=f"r = {corr_i:.3f}",
        xref=f"x{i}", yref=f"y{i}",
        x=lo + 0.15 * (hi - lo), y=hi - 0.05 * (hi - lo),
        showarrow=False, font=dict(size=11, color="red"),
    )

fig.update_layout(
    title_text=f"Distance Preservation vs Projection Dimension (d={d}, n={n})",
    template="plotly_white",
    height=400, width=1100,
)
fig.show()

print("\nMore projection dimensions -> tighter distances -> better preservation.")
print("But even m=50 (10x compression) is remarkably good!")

## 3. The Johnson-Lindenstrauss Lemma (Intuition, Not Proof)

What we just observed is not a coincidence. It is guaranteed by a beautiful result from 1984:

> **JL Lemma**: For any set of $n$ points in $\mathbb{R}^d$ and any $\epsilon > 0$, there exists a projection into $m = O\left(\frac{\log n}{\epsilon^2}\right)$ dimensions such that **all** pairwise distances are preserved within a $(1 \pm \epsilon)$ factor.

Concretely, for all pairs $i, j$:

$$
(1 - \epsilon) \, \|x_i - x_j\|^2 \;\leq\; \|y_i - y_j\|^2 \;\leq\; (1 + \epsilon) \, \|x_i - x_j\|^2
$$

### What makes this amazing

- $m$ depends on $\log(n)$, **not** on $d$. Even if $d = 1{,}000{,}000$, you only need $O(\log n)$ dimensions.
- The projection is **data-oblivious** -- a random Gaussian matrix works, no training needed.
- The result is **universal**: it works for ANY set of $n$ points.

### Why it works (intuition)

In high dimensions, random directions are **nearly orthogonal** to each other. So when you project onto $m$ random directions, you're effectively "sampling" the geometry from $m$ independent viewpoints. As $m$ grows, the average of these samples concentrates around the true distance.

In [ ]:
section_header(
    "The JL Lemma",
    "m = O(log(n) / epsilon^2) dimensions suffice"
)

# The JL bound: m >= 4 * ln(n) / epsilon^2
# (the constant 4 comes from a standard proof using sub-Gaussian concentration)

def jl_bound(n, epsilon):
    """Minimum projection dimension from the JL lemma."""
    return int(math.ceil(4 * math.log(n) / epsilon**2))

# Show how m scales with n and epsilon
print(f"{'n':>10}  {'epsilon':>8}  {'m (JL bound)':>14}  {'Note':>30}")
print("-" * 68)
for n_pts in [100, 1000, 10_000, 100_000, 1_000_000]:
    for eps in [0.1, 0.2, 0.5]:
        m_jl = jl_bound(n_pts, eps)
        note = "" if n_pts < 1_000_000 else "<-- 1M points!"
        print(f"{n_pts:>10,}  {eps:>8.1f}  {m_jl:>14,}  {note:>30}")
    print()

print("Key observation: going from 100 to 1,000,000 points only ~doubles m.")
print("That's the power of log(n).")

### Let's verify the $(1 \pm \epsilon)$ guarantee empirically

For each pair, we compute the **distortion ratio**:

$$
\text{ratio} = \frac{\|y_i - y_j\|^2}{\|x_i - x_j\|^2}
$$

The JL lemma says this should be in $[1 - \epsilon, 1 + \epsilon]$ for all pairs.

In [ ]:
# Use our earlier X (n=200, d=512)
# Try several m values and check what fraction of pairs violate the epsilon bound

epsilon = 0.2
m_test_values = [10, 25, 50, 100, 200, 400]

true_sq = true_dists[mask] ** 2

results = []
for m_val in m_test_values:
    R_test = torch.randn(m_val, d) / math.sqrt(m_val)
    Y_test = X @ R_test.t()
    proj_sq = torch.cdist(Y_test, Y_test)[mask] ** 2

    ratios = proj_sq / true_sq
    violations = ((ratios < 1 - epsilon) | (ratios > 1 + epsilon)).float().mean().item()
    max_distortion = (ratios - 1).abs().max().item()
    results.append((m_val, violations, max_distortion))

    m_jl = jl_bound(n, epsilon)

print(f"JL bound for n={n}, epsilon={epsilon}: m >= {jl_bound(n, epsilon)}")
print(f"\n{'m':>6}  {'Violation %':>12}  {'Max |ratio-1|':>14}  {'Status':>10}")
print("-" * 48)
for m_val, viol, max_d in results:
    status = "PASS" if viol == 0.0 else "some violations"
    print(f"{m_val:>6}  {viol*100:>11.1f}%  {max_d:>14.3f}  {status:>10}")

## 4. Why Gaussian Random Matrices Work

Each row of $R$ is a random direction in $\mathbb{R}^d$. Let's understand why projecting onto random directions preserves distances.

**Key identity**: If $r \sim \mathcal{N}(0, I_d)$ and $x \in \mathbb{R}^d$, then:

$$
\mathbb{E}\left[(r \cdot x)^2\right] = \|x\|^2
$$

This says: the squared projection onto a random direction is an **unbiased estimate** of the squared norm.

When we project onto $m$ random directions and average (the $1/\sqrt{m}$ scaling), the law of large numbers kicks in: the estimate **concentrates** around the true value.

In [ ]:
section_header(
    "Why Random Projections Work",
    "E[<r, x>^2] = ||x||^2  --  unbiased norm estimation"
)

# Demonstrate the key identity
torch.manual_seed(123)
x = torch.randn(d)  # a single vector
true_norm_sq = x.norm() ** 2

# Project onto many random directions and check
n_trials = 10000
r = torch.randn(n_trials, d)
projections = (r @ x) ** 2   # each is (r_i . x)^2

print(f"True ||x||^2 = {true_norm_sq.item():.2f}")
print(f"E[(r . x)^2] = {projections.mean().item():.2f}  (over {n_trials:,} random directions)")
print(f"\nThey match! Each random projection gives an unbiased estimate of the norm.")
print(f"\nNow averaging over m directions concentrates the estimate:")

# Show concentration as m increases
m_vals = [1, 5, 10, 50, 100, 500]
n_experiments = 1000

stds = []
for m_val in m_vals:
    estimates = []
    for _ in range(n_experiments):
        R_exp = torch.randn(m_val, d)
        proj_norm_sq = (R_exp @ x).pow(2).mean()  # average over m directions
        estimates.append(proj_norm_sq.item())
    std = np.std(estimates)
    stds.append(std)
    print(f"  m = {m_val:>4}: std of estimate = {std:.3f}  (relative: {std / true_norm_sq.item():.3f})")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=m_vals, y=stds,
    mode="lines+markers",
    marker=dict(size=8, color="steelblue"),
    line=dict(width=2),
))
fig.update_layout(
    title="Concentration: Std of Norm Estimate vs Number of Projection Dimensions",
    xaxis_title="Projection dimensions (m)",
    yaxis_title="Std of norm estimate",
    template="plotly_white",
    height=400, width=600,
)
fig.show()

## 5. Inner Product Preservation

Preserving distances is great, but for TurboQuant we specifically need to preserve **inner products** (for attention scores, similarity search, etc.).

Good news: distance preservation **implies** inner product preservation. Here's why:

$$
\|a - b\|^2 = \|a\|^2 + \|b\|^2 - 2\langle a, b \rangle
$$

So:

$$
\langle a, b \rangle = \frac{1}{2}\left(\|a\|^2 + \|b\|^2 - \|a - b\|^2\right)
$$

If the projection preserves all three norms on the right-hand side, it preserves the inner product too.

Let's verify directly.

In [ ]:
section_header(
    "Inner Product Preservation",
    "Crucial for attention and search"
)

torch.manual_seed(42)
m_ip = 100

R_ip = torch.randn(m_ip, d) / math.sqrt(m_ip)
Y_ip = X @ R_ip.t()

# True inner products (all pairs)
true_ip = X @ X.t()             # (n, n)
proj_ip = Y_ip @ Y_ip.t()       # (n, n)

# Extract upper triangle
true_ip_flat = true_ip[mask].numpy()
proj_ip_flat = proj_ip[mask].numpy()

sample = rng.choice(len(true_ip_flat), size=3000, replace=False)
corr_ip = np.corrcoef(true_ip_flat, proj_ip_flat)[0, 1]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=true_ip_flat[sample], y=proj_ip_flat[sample],
    mode="markers",
    marker=dict(size=3, opacity=0.4, color="darkorange"),
    name="Pairs",
))

ip_lo, ip_hi = true_ip_flat.min(), true_ip_flat.max()
fig.add_trace(go.Scatter(
    x=[ip_lo, ip_hi], y=[ip_lo, ip_hi],
    mode="lines", line=dict(color="red", dash="dash", width=2),
    name="Perfect",
))

fig.update_layout(
    title=f"Inner Product Preservation: d={d} -> m={m_ip}  (correlation = {corr_ip:.4f})",
    xaxis_title="True inner product",
    yaxis_title="Projected inner product",
    template="plotly_white",
    height=500, width=600,
)
fig.show()

print(f"Inner product correlation: {corr_ip:.4f}")
print("\nThis is exactly what TurboQuant needs: preserved inner products")
print("for computing attention scores and nearest-neighbor search.")

## 6. Build a Simple Random Projection Module

Let's package this into a reusable class. The key design choices:

1. **Seeded random matrix**: use a fixed seed so both encoder and decoder share the same $R$ without transmitting it.
2. **Scaling by $1/\sqrt{m}$**: baked into the matrix so `project()` is a simple matmul.
3. **No learnable parameters**: this is entirely data-oblivious.

In [ ]:
section_header(
    "Building a Random Projection Module",
    "Reusable, seeded, data-oblivious"
)


class RandomProjection:
    """Random Gaussian projection for dimensionality reduction.

    The projection matrix R is seeded, so both encoder and decoder
    can reconstruct it from the seed alone -- no need to store or
    transmit the matrix.
    """

    def __init__(self, input_dim, proj_dim, seed=42):
        gen = torch.Generator()
        gen.manual_seed(seed)
        self.R = torch.randn(proj_dim, input_dim, generator=gen) / math.sqrt(proj_dim)
        self.input_dim = input_dim
        self.proj_dim = proj_dim

    def project(self, x):
        """Project x from input_dim to proj_dim.

        Args:
            x: tensor of shape (..., input_dim)

        Returns:
            tensor of shape (..., proj_dim)
        """
        return x @ self.R.t()

    def __repr__(self):
        return f"RandomProjection({self.input_dim} -> {self.proj_dim})"


# Quick test
rp = RandomProjection(input_dim=512, proj_dim=64)
print(rp)

test_batch = torch.randn(32, 512)
projected = rp.project(test_batch)
print(f"Input shape:  {test_batch.shape}")
print(f"Output shape: {projected.shape}")

In [ ]:
# Verify norm preservation with the module
rp = RandomProjection(input_dim=512, proj_dim=128, seed=0)

X_test = torch.randn(500, 512)
Y_test = rp.project(X_test)

original_norms = X_test.norm(dim=1)
projected_norms = Y_test.norm(dim=1)

rel_error = ((projected_norms - original_norms) / original_norms).abs()
print(f"Norm preservation (m=128, d=512):")
print(f"  Mean relative error: {rel_error.mean():.4f}")
print(f"  Max  relative error: {rel_error.max():.4f}")
print(f"  Std  relative error: {rel_error.std():.4f}")

# Inner products
true_ips = (X_test[:100] @ X_test[:100].t())
proj_ips = (Y_test[:100] @ Y_test[:100].t())
mask100 = torch.triu(torch.ones(100, 100, dtype=torch.bool), diagonal=1)
ip_err = ((proj_ips[mask100] - true_ips[mask100]) / true_ips[mask100].abs().clamp(min=1.0)).abs()
print(f"\nInner product preservation:")
print(f"  Mean relative error: {ip_err.mean():.4f}")
print(f"  Max  relative error: {ip_err.max():.4f}")

## 7. The Setup for QJL

We have shown that random projection to $m$ dimensions preserves distances and inner products. Each projected coordinate is a float32 value -- **32 bits**.

But here is the radical question:

> **What if we kept only the SIGN of each projected coordinate?**
>
> That's **1 bit** per dimension instead of 32.

This would give us a 32x further compression on top of the dimensionality reduction. It sounds absurd -- surely throwing away the magnitude destroys the geometry?

Surprisingly, it doesn't. The **sign** of a random projection contains a remarkable amount of information about the original inner product. This is the core idea behind **QJL (Quantized Johnson-Lindenstrauss)**, which is the subject of Module 08.

In [ ]:
section_header(
    "Preview: What if we only keep the sign?",
    "1 bit per dimension -- that's Module 08"
)

# Quick preview of the sign-only idea
rp_preview = RandomProjection(input_dim=512, proj_dim=256, seed=7)

X_prev = torch.randn(100, 512)
Y_prev = rp_preview.project(X_prev)
S_prev = torch.sign(Y_prev)  # keep only the sign!

# Compare inner products
true_ip_prev = X_prev @ X_prev.t()
sign_ip_prev = S_prev @ S_prev.t() / 256  # scale back

mask_prev = torch.triu(torch.ones(100, 100, dtype=torch.bool), diagonal=1)
t = true_ip_prev[mask_prev].numpy()
s = sign_ip_prev[mask_prev].numpy()

# The sign-only inner product is related to the true one, but with a nonlinear transform
# (it's actually estimating the cosine angle via the arccosine kernel)
corr_sign = np.corrcoef(t, s)[0, 1]
print(f"Correlation between true inner products and sign-only inner products: {corr_sign:.3f}")
print(f"\nNot bad for 1-bit projections! But there's a bias we need to fix.")
print(f"Module 08 shows how QJL makes this estimate UNBIASED with a clever scaling.")

---

## Exercise: JL Bound Verification

**Setup**: You have $n = 100$ vectors in $d = 256$ dimensions.

**Task**: What is the minimum projection dimension $m$ to keep all pairwise distance distortions below 20% ($\epsilon = 0.2$)?

Use the JL bound:

$$
m = \left\lceil \frac{4 \ln(n)}{\epsilon^2} \right\rceil
$$

1. Compute the theoretical bound.
2. Verify empirically: project 100 vectors from $d = 256$ to your computed $m$, and check that the maximum distortion ratio is within $[0.8, 1.2]$.

In [ ]:
section_header("Exercise", "JL bound verification")

n_ex = 100
d_ex = 256
epsilon_ex = 0.2

# ---- Step 1: Compute the JL bound ----
# YOUR CODE HERE: compute m_theory using the formula above
m_theory = ...  # FILL IN

print(f"For n={n_ex}, epsilon={epsilon_ex}:")
print(f"  Theoretical JL bound: m >= {m_theory}")

In [ ]:
# Check your answer
check_intuition(
    m_theory,
    expected=461,
    hint="m = ceil(4 * ln(100) / 0.2^2). Remember: ln, not log10.",
    label="JL bound m",
    tolerance=0.01,
)

In [ ]:
# ---- Step 2: Empirical verification ----
# YOUR CODE HERE:
# 1. Create 100 random vectors in d=256
# 2. Project to m_theory dimensions using a random Gaussian matrix
# 3. Compute pairwise distance ratios (projected^2 / true^2)
# 4. Check that all ratios are in [1-epsilon, 1+epsilon]

torch.manual_seed(0)

# Your code here
# X_ex = ...
# R_ex = ...
# Y_ex = ...
# ... compute ratios ...
# max_distortion = ...

# Uncomment when ready:
# print(f"Max distortion: {max_distortion:.4f}")
# print(f"Within epsilon={epsilon_ex}? {max_distortion < epsilon_ex}")

In [ ]:
# ---- Solution (run after attempting) ----

torch.manual_seed(0)

X_ex = torch.randn(n_ex, d_ex)
R_ex = torch.randn(461, d_ex) / math.sqrt(461)
Y_ex = X_ex @ R_ex.t()

true_d_ex = torch.cdist(X_ex, X_ex)
proj_d_ex = torch.cdist(Y_ex, Y_ex)

mask_ex = torch.triu(torch.ones(n_ex, n_ex, dtype=torch.bool), diagonal=1)
ratios_ex = (proj_d_ex[mask_ex] ** 2) / (true_d_ex[mask_ex] ** 2)
max_distortion = (ratios_ex - 1).abs().max().item()

print(f"Using m = 461 (the JL bound):")
print(f"  Max |ratio - 1| = {max_distortion:.4f}")
print(f"  Within epsilon={epsilon_ex}? {max_distortion < epsilon_ex}")
print(f"  Number of violations: {((ratios_ex < 1 - epsilon_ex) | (ratios_ex > 1 + epsilon_ex)).sum().item()} / {mask_ex.sum().item()}")

# Histogram of distortion ratios
fig = go.Figure()
fig.add_trace(go.Histogram(
    x=ratios_ex.numpy(),
    nbinsx=80,
    marker_color="steelblue",
    opacity=0.7,
))
fig.add_vline(x=1 - epsilon_ex, line_dash="dash", line_color="red",
              annotation_text=f"1 - eps = {1-epsilon_ex}")
fig.add_vline(x=1 + epsilon_ex, line_dash="dash", line_color="red",
              annotation_text=f"1 + eps = {1+epsilon_ex}")
fig.add_vline(x=1.0, line_color="green", line_width=2)
fig.update_layout(
    title="Distribution of Pairwise Distance Ratios (projected / true)",
    xaxis_title="Ratio (projected^2 / true^2)",
    yaxis_title="Count",
    template="plotly_white",
    height=400, width=700,
)
fig.show()

---

## Summary

| Concept | Key takeaway |
|---------|-------------|
| Random projection | Multiply by a random matrix to reduce dimension |
| JL Lemma | $m = O(\log n / \epsilon^2)$ dims preserve all pairwise distances within $(1 \pm \epsilon)$ |
| $m$ depends on $\log n$ | Even millions of points need only hundreds of dimensions |
| Data-oblivious | No training, no data access -- just a seeded random matrix |
| Inner products preserved | Because they can be recovered from distances |
| Why it works | Random directions are nearly orthogonal in high-d; averaging concentrates |

### What's next

**Module 08**: We take this to the extreme -- keep only the **sign** of each projection (1 bit per dimension). This is **QJL (Quantized Johnson-Lindenstrauss)**, and it gives unbiased inner product estimates despite discarding almost all information. That's the second ingredient TurboQuant needs.